<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 27 · Valuation Framework

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by various LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the code examples from the chapter in a Colab-ready
format so that you can run, tweak, and extend them interactively.


### How to Use This Notebook
- Run the cells top to bottom the first time to create all variables.
- Use additional cells for your own experiments or GenAI-assisted
  refactorings.
- Refer back to the book text for detailed explanations and context.


In [ ]:
from pathlib import Path
import subprocess
import sys

NOTEBOOK_SUBDIR = "notebooks"
COLAB_PACKAGES = {}
REPO_NAME = "py4fi3rd"
REPO_URL = "https://github.com/yhilpisch/py4fi3rd.git"


def _support_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        support_dir = candidate / "notebooks"
        if (support_dir / "_book_notebook_support.py").exists():
            return support_dir
    if "google.colab" in sys.modules:
        root = Path("/content") / REPO_NAME
        if not root.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(root)],
                check=True,
            )
        return root / "notebooks"
    raise RuntimeError("Could not locate notebook support helpers.")


SUPPORT_DIR = _support_dir()
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))

from _book_notebook_support import setup_notebook

CONTEXT = setup_notebook(
    notebook_subdir=NOTEBOOK_SUBDIR,
    colab_packages=COLAB_PACKAGES,
)

PROJECT_ROOT = CONTEXT["PROJECT_ROOT"]
NOTEBOOK_DIR = CONTEXT["NOTEBOOK_DIR"]
CODE_DIR = CONTEXT["CODE_DIR"]
CHAPTERS_DIR = CONTEXT["CHAPTERS_DIR"]
FIGURES_DIR = CONTEXT["FIGURES_DIR"]
DATA_DIR = CONTEXT["DATA_DIR"]

PROJECT_ROOT

## From Risk-Neutral Valuation to a Software Architecture

Review the mathematical components of risk-neutral valuation that drive the
software architecture requirements.


## Design Goals and Library Layout (`dxlib`)

The `dxlib` library separates valuation tasks into single-responsibility
modules to maximize testability and reuse.

## Time Handling and Year Fractions

Standardize time grids and year-fraction calculations to prevent
inconsistencies across pricing routines.


In [ ]:

import datetime as dt

import numpy as np


def ensure_datetime_array(times: list[dt.date | dt.datetime]) -> np.ndarray:
    normalized: list[dt.datetime] = []
    for value in times:
        if isinstance(value, dt.datetime):
            # Preserve datetime.datetime values as-is.
            normalized.append(value)
        elif isinstance(value, dt.date):
            # Promote datetime.date to datetime.datetime so
            # arithmetic is consistent.
            normalized.append(dt.datetime.combine(value, dt.time()))
        else:
            raise TypeError("Unsupported time entry")
    if not normalized:
        raise ValueError("times iterable is empty")
    # Sort the time points to make downstream calculations deterministic.
    return np.array(sorted(normalized), dtype=object)


def year_fractions(
    times: list[dt.date | dt.datetime],
    day_count: float = 365.0,
) -> np.ndarray:
    # Normalize and order the grid before computing deltas.
    ordered = ensure_datetime_array(times)
    # Use the earliest time point as the origin (pricing date in most
    # use cases).
    origin = ordered[0]
    # Compute ACT/day_count year fractions.
    deltas = [
        (ts - origin).total_seconds() / 86_400.0 / day_count for ts in ordered
    ]
    return np.array(deltas, dtype=float)

## Deterministic Discounting as a Curve Object

Wrap scalar discount rates into curve objects that expose a uniform
`discount_factor` interface.


In [ ]:
import sys
from pathlib import Path

# Add the code directory to sys.path so dxlib can be imported
CODE_DIR = (Path("..") / "code").resolve()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import datetime as dt
import math
from dataclasses import dataclass

from dxlib.time import time_to_maturity


@dataclass(frozen=True, slots=True)
class ConstantShortRateCurve:
    name: str
    reference_date: dt.date | dt.datetime
    rate: float
    day_count: float = 365.0

    def discount_factor(self, target: dt.date | dt.datetime) -> float:
        # Convert the maturity date into a year fraction using the
        # shared time helper.
        ttm = time_to_maturity(
            self.reference_date,
            target,
            day_count=self.day_count,
        )
        # Apply continuous compounding to get the discount factor.
        return float(math.exp(-self.rate * ttm))

## Market Environments and Dependency Injection

Bundle constants, lists, and discount curves into a single `MarketEnvironment`
container to simplify pricing functions.


In [ ]:
from dataclasses import dataclass, field
from typing import Iterable, MutableMapping

from dxlib.curves import DiscountCurve


@dataclass
class MarketEnvironment:
    name: str
    pricing_date: dt.date | dt.datetime
    # Constants capture scalar inputs reused across models and pricers.
    constants: MutableMapping[str, float] = field(default_factory=dict)
    # Lists capture collections such as basket constituents.
    lists: MutableMapping[str, list[str]] = field(default_factory=dict)
    # Curves store discounting objects behind a small interface.
    curves: MutableMapping[str, DiscountCurve] = field(default_factory=dict)

    def add_constant(self, key: str, value: float) -> None:
        self.constants[key] = float(value)

    def add_list(self, key: str, values: Iterable[str]) -> None:
        self.lists[key] = list(values)

    def add_curve(self, key: str, curve: DiscountCurve) -> None:
        self.curves[key] = curve

## Deterministic by Default, Interchangeable by Design

Pricers consume discounting objects through a minimal interface, allowing
deterministic curves to be swapped for stochastic ones later.


## A Quick Interactive Sanity Check

Add the local code directory to the system path and verify that the core
`dxlib` components can be instantiated.


In [ ]:
import sys
from pathlib import Path

# Point to the code/ directory relative to the current working directory.
CODE_DIR = (Path("..") / "code").resolve()

# Prepend code/ to sys.path so dxlib becomes importable.
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import datetime as dt
from dxlib import (
    ConstantShortRateCurve,
    InterpolatedZeroCurve,
    MarketEnvironment,
)

In [ ]:
pricing_date = dt.date(2026, 1, 26)
maturity = dt.date(2026, 7, 26)

# Instantiate a constant short-rate discount curve.
curve = ConstantShortRateCurve(
    "r",
    reference_date=pricing_date,
    rate=0.03,
)

# Compute the discount factor for the maturity date.
curve.discount_factor(maturity)

In [ ]:
curve_term = InterpolatedZeroCurve(
    "r_term",
    reference_date=pricing_date,
    nodes=[
        (dt.date(2026, 7, 26), 0.03),
        (dt.date(2027, 1, 26), 0.035),
    ],
)
round(curve_term.discount_factor(dt.date(2026, 10, 26)), 4)

In [ ]:
# Instantiate a market environment for this valuation context.
env = MarketEnvironment("spx", pricing_date=pricing_date)
# Add a constant parameter to the environment.
env.add_constant("volatility", 0.20)
# Add the discount curve under a clear key.
env.add_curve("discount_curve", curve)

# Retrieve the stored constant to confirm the interface works.
env.get_constant("volatility")

In [ ]:
overlay = MarketEnvironment("spx-overlay", pricing_date=pricing_date)
overlay.add_constant("paths", 50_000)
env.merge(overlay)
sorted(env.snapshot()["constants"])

## Where We Are Heading Next

With time, discounting, and market environments in place, the next step is to
build stochastic simulators for risk factors.


## Appendix: dxlib Foundation Source Code

The complete source code for the foundational `dxlib` modules is included below
for reference.

### code/dxlib/time.py


In [ ]:
"""Python for Finance, 3rd ed., O'Reilly (2026).
Chapter 27 - Valuation Framework.

Time-axis utilities shared across simulation, valuation, and discounting.

(c) Dr. Yves J. Hilpisch
AI-supported by various LLMs
The Python Quants GmbH | https://tpq.io
https://hilpisch.com | https://linktr.ee/dyjh
"""

from __future__ import annotations

import datetime as dt
from collections.abc import Iterable, Sequence
from typing import overload

import numpy as np

__all__ = ["ensure_datetime_array", "year_fractions", "time_to_maturity"]


@overload
def ensure_datetime_array(times: Sequence[dt.datetime]) -> np.ndarray: ...


@overload
def ensure_datetime_array(times: Sequence[dt.date]) -> np.ndarray: ...


def ensure_datetime_array(
    times: Sequence[dt.date | dt.datetime],
) -> np.ndarray:
    """
    Convert ``datetime``/``date`` objects to a sorted NumPy array.

    The array uses dtype ``object`` and contains ``datetime.datetime`` objects.
    """

    if not isinstance(times, Iterable):
        raise TypeError("times must be an iterable of datetime/date objects")

    normalized: list[dt.datetime] = []
    for value in times:
        if isinstance(value, dt.datetime):
            normalized.append(value)
        elif isinstance(value, dt.date):
            normalized.append(dt.datetime.combine(value, dt.time()))
        else:  # pragma: no cover
            msg = (
                "Unsupported time entry "
                f"{value!r} (type {type(value).__name__})"
            )
            raise TypeError(msg)

    if not normalized:
        raise ValueError("times iterable is empty")

    return np.array(sorted(normalized), dtype=object)


def year_fractions(
    times: Sequence[dt.date | dt.datetime],
    day_count: float = 365.0,
) -> np.ndarray:
    """
    Compute ACT/day_count year fractions relative to the first
    (earliest) timestamp.
    """

    if day_count <= 0:
        raise ValueError("day_count must be strictly positive")

    ordered = ensure_datetime_array(times)
    origin = ordered[0]
    deltas: list[float] = []
    for ts in ordered:
        days = (ts - origin).total_seconds() / 86_400.0
        deltas.append(days / day_count)
    return np.array(deltas, dtype=float)


def time_to_maturity(
    pricing_date: dt.date | dt.datetime,
    maturity: dt.date | dt.datetime,
    *,
    day_count: float = 365.0,
) -> float:
    """
    Convenience helper returning the ACT/day_count time-to-maturity in years.
    """

    if day_count <= 0:
        raise ValueError("day_count must be strictly positive")

    dates = []
    for value in (pricing_date, maturity):
        if isinstance(value, dt.datetime):
            dates.append(value)
        elif isinstance(value, dt.date):
            dates.append(dt.datetime.combine(value, dt.time()))
        else:  # pragma: no cover
            msg = (
                "Unsupported time entry "
                f"{value!r} (type {type(value).__name__})"
            )
            raise TypeError(msg)

    delta_days = (dates[1] - dates[0]).total_seconds() / 86_400.0
    if delta_days < 0:
        raise ValueError("maturity must be on or after pricing_date")
    return float(delta_days / day_count)

### code/dxlib/curves.py


In [ ]:
"""Python for Finance, 3rd ed., O'Reilly (2026).
Chapter 27 - Valuation Framework.

Discount curve building blocks.

(c) Dr. Yves J. Hilpisch
AI-supported by various LLMs
The Python Quants GmbH | https://tpq.io
https://hilpisch.com | https://linktr.ee/dyjh
"""

from __future__ import annotations

__package__ = "dxlib"

import datetime as dt
import math
import sys
from collections.abc import Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Protocol

import numpy as np

if __name__ == "__main__" and __package__ is None:
    package_dir = Path(__file__).resolve().parent
    sys.path = [
        path
        for path in sys.path
        if Path(path or ".").resolve() != package_dir
    ]
    sys.path.insert(0, str(package_dir.parent))
    __package__ = "dxlib"

from .time import time_to_maturity

__all__ = ["DiscountCurve", "ConstantShortRateCurve", "InterpolatedZeroCurve"]


class DiscountCurve(Protocol):
    """
    Minimal discounting interface used throughout Part VI.

    Deterministic curves and stochastic discounting models can both implement
    this protocol.
    """

    reference_date: dt.date | dt.datetime

    def discount_factor(self, target: dt.date | dt.datetime) -> float: ...


@dataclass(frozen=True, slots=True)
class ConstantShortRateCurve:
    """
    Constant continuously compounded short-rate curve.
    """

    name: str
    reference_date: dt.date | dt.datetime
    rate: float
    day_count: float = 365.0

    def __post_init__(self) -> None:
        if self.day_count <= 0:
            raise ValueError("day_count must be positive")

    def discount_factor(self, target: dt.date | dt.datetime) -> float:
        ttm = time_to_maturity(
            self.reference_date,
            target,
            day_count=self.day_count,
        )
        return float(math.exp(-self.rate * ttm))


@dataclass(frozen=True)
class InterpolatedZeroCurve:
    """
    Deterministic zero curve represented by (date, zero_rate) nodes.

    Linear interpolation is applied to continuously compounded zero
    rates.
    """

    name: str
    reference_date: dt.date | dt.datetime
    nodes: Sequence[tuple[dt.date | dt.datetime, float]]
    day_count: float = 365.0
    extrapolate: str = "flat"

    def __post_init__(self) -> None:
        if self.day_count <= 0:
            raise ValueError("day_count must be positive")
        if len(self.nodes) < 2:
            raise ValueError(
                "InterpolatedZeroCurve requires at least two nodes"
            )
        mode = str(self.extrapolate).strip().lower()
        if mode not in {"flat", "error"}:
            raise ValueError("extrapolate must be 'flat' or 'error'")
        times, _ = self._times_and_rates()
        if len(np.unique(times)) != times.size:
            raise ValueError("curve node maturities must be unique")

    def _times_and_rates(self) -> tuple[np.ndarray, np.ndarray]:
        ordered = sorted(self.nodes, key=lambda pair: pair[0])
        dates, rates = zip(*ordered)
        times = np.array(
            [
                time_to_maturity(
                    self.reference_date,
                    d,
                    day_count=self.day_count,
                )
                for d in dates
            ],
            dtype=float,
        )
        return times, np.array(rates, dtype=float)

    def zero_rate(self, target: dt.date | dt.datetime) -> float:
        if target == self.reference_date:
            return 0.0
        times, rates = self._times_and_rates()
        t = time_to_maturity(
            self.reference_date,
            target,
            day_count=self.day_count,
        )
        if t <= 0:
            raise ValueError("target must be after reference_date")
        mode = str(self.extrapolate).strip().lower()
        if mode == "error" and (t < float(times[0]) or t > float(times[-1])):
            raise ValueError("target is outside curve node range")
        r = float(np.interp(t, times, rates))
        return r

    def discount_factor(self, target: dt.date | dt.datetime) -> float:
        if target == self.reference_date:
            return 1.0
        t = time_to_maturity(
            self.reference_date,
            target,
            day_count=self.day_count,
        )
        r = self.zero_rate(target)
        return float(math.exp(-r * t))

### code/dxlib/env.py


In [ ]:
"""Python for Finance, 3rd ed., O'Reilly (2026).
Chapter 27 - Valuation Framework.

Market environment container.

(c) Dr. Yves J. Hilpisch
AI-supported by various LLMs
The Python Quants GmbH | https://tpq.io
https://hilpisch.com | https://linktr.ee/dyjh
"""

from __future__ import annotations

__package__ = "dxlib"

import datetime as dt
import sys
from collections.abc import Iterable, Mapping, MutableMapping
from dataclasses import dataclass, field
from pathlib import Path

if __name__ == "__main__" and __package__ is None:
    package_dir = Path(__file__).resolve().parent
    sys.path = [
        path
        for path in sys.path
        if Path(path or ".").resolve() != package_dir
    ]
    sys.path.insert(0, str(package_dir.parent))
    __package__ = "dxlib"

from .curves import DiscountCurve

__all__ = ["MarketEnvironment"]


@dataclass
class MarketEnvironment:
    """
    Container bundling pricing date, constants, lists, and curves.

    The design mirrors what many derivatives libraries do in practice:
    instead of
    passing dozens of arguments to every pricer, you pass a single environment
    object and extract what you need.
    """

    name: str
    pricing_date: dt.date | dt.datetime
    constants: MutableMapping[str, float] = field(default_factory=dict)
    lists: MutableMapping[str, list[str]] = field(default_factory=dict)
    curves: MutableMapping[str, DiscountCurve] = field(default_factory=dict)

    def add_constant(self, key: str, value: float) -> None:
        self.constants[key] = float(value)

    def get_constant(self, key: str) -> float:
        return float(self.constants[key])

    def add_list(self, key: str, values: Iterable[str]) -> None:
        self.lists[key] = list(values)

    def get_list(self, key: str) -> list[str]:
        return list(self.lists[key])

    def add_curve(self, key: str, curve: DiscountCurve) -> None:
        self.curves[key] = curve

    def get_curve(self, key: str) -> DiscountCurve:
        return self.curves[key]

    def merge(self, other: MarketEnvironment) -> None:
        """
        Merge another environment into this one, overriding duplicate keys.
        """

        self.constants.update(other.constants)
        self.lists.update(other.lists)
        self.curves.update(other.curves)

    def snapshot(self) -> Mapping[str, Mapping[str, object]]:
        """
        Return an immutable snapshot for logging/debugging.
        """

        return {
            "constants": dict(self.constants),
            "lists": {key: list(values) for key, values in self.lists.items()},
            "curves": dict(self.curves),
        }

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
